<a href="https://colab.research.google.com/github/gouthamsankar-clg/AI-Lab/blob/main/Lab_8/FOL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
%pip -q install ipywidgets matplotlib

import math
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
from matplotlib.patches import Arc

%matplotlib inline


# ---------- PARSING ----------

def parse_atom(s):
    s = s.strip()
    if "(" not in s:
        return s, ()
    predicate, args = s.split("(", 1)
    if not args.endswith(")"):
        raise ValueError("Invalid predicate: " + s)
    args = args[:-1].strip()
    if not args:
        return predicate.strip(), ()
    return predicate.strip(), tuple(x.strip() for x in args.split(","))


def parse_rule(s):
    if "->" not in s:
        raise ValueError("Rule must contain ->")
    left, right = s.split("->", 1)
    premises = [parse_atom(x.strip()) for x in left.split("&")]
    return premises, parse_atom(right.strip())


def is_variable(x):
    return bool(x) and x[0].islower()


def format_atom(atom):
    predicate, args = atom
    return predicate if not args else predicate + "(" + ", ".join(args) + ")"


# ---------- UNIFICATION ----------

def match(pattern, fact, substitution):
    if pattern[0] != fact[0] or len(pattern[1]) != len(fact[1]):
        return None
    substitution = substitution.copy()
    for p, f in zip(pattern[1], fact[1]):
        if is_variable(p):
            if p in substitution:
                if substitution[p] != f:
                    return None
            else:
                substitution[p] = f
        elif p != f:
            return None
    return substitution


def apply_substitution(atom, substitution):
    predicate, args = atom
    return predicate, tuple(
        substitution.get(x, x) if is_variable(x) else x for x in args
    )


# ---------- FORWARD CHAINING ----------

def forward_chaining(facts, rules, goal):
    known = set(facts)
    goal = parse_atom(goal)
    derivations = []
    step = 1

    print("Initial Facts:")
    for fact in sorted(known, key=format_atom):
        print(" •", format_atom(fact))

    while True:
        new_facts = set()
        candidates = []

        for rule_index, (premises, conclusion) in enumerate(rules):
            substitutions = [({}, [])]
            for premise in premises:
                temp = []
                for subst, matched in substitutions:
                    for fact in sorted(known, key=format_atom):
                        result = match(premise, fact, subst)
                        if result is not None:
                            temp.append((result, matched + [fact]))
                substitutions = temp
                if not substitutions:
                    break

            for subst, matched in substitutions:
                inferred = apply_substitution(conclusion, subst)
                if inferred not in known:
                    new_facts.add(inferred)
                    candidates.append({
                        "premises": matched,
                        "conclusion": inferred,
                        "rule": rule_index,
                        "step": step,
                    })

        if not new_facts:
            break

        print(f"\nInference Step {step}")
        for fact in sorted(new_facts, key=format_atom):
            print("  →", format_atom(fact))

        # keep every distinct (rule, premises) derivation of this step
        seen = set()
        for d in candidates:
            key = (d["rule"], tuple(d["premises"]), d["conclusion"])
            if key not in seen:
                seen.add(key)
                derivations.append(d)

        known.update(new_facts)
        step += 1

    print("\nFinal Knowledge Base:")
    for fact in sorted(known, key=format_atom):
        print(" •", format_atom(fact))

    print("\nGoal:", format_atom(goal))
    print("✓ GOAL PROVED" if goal in known else "✗ GOAL NOT PROVED")

    draw_graph(facts, derivations, goal, known)
    return known


# ---------- SLIDE-STYLE AND GRAPH ----------

def draw_graph(initial_facts, derivations, goal, known):
    X_GAP, Y_GAP = 2.0, 2.4

    level = {}
    for f in initial_facts:
        level[f] = 0
    for d in derivations:
        level[d["conclusion"]] = d["step"]

    # --- x positions ---
    pos = {}
    for i, f in enumerate(initial_facts):
        pos[f] = i * X_GAP

    max_level = max(level.values())
    for lv in range(1, max_level + 1):
        nodes = [n for n in level if level[n] == lv]
        desired = {}
        for n in nodes:
            xs = [pos[p] for d in derivations if d["conclusion"] == n
                  for p in d["premises"]]
            desired[n] = sum(xs) / len(xs)
        nodes.sort(key=lambda n: desired[n])
        last = None
        for n in nodes:
            x = desired[n]
            if last is not None and x < last + X_GAP:
                x = last + X_GAP
            pos[n] = x
            last = x

    # center the whole drawing around the goal column
    coords = {n: (pos[n], level[n] * Y_GAP) for n in level}

    xs = [c[0] for c in coords.values()]
    ys = [c[1] for c in coords.values()]
    width = max(6, (max(xs) - min(xs)) * 0.9 + 3)
    height = max(5, (max(ys) - min(ys)) * 0.9 + 2.5)

    fig, ax = plt.subplots(figsize=(width, height))
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_xlim(min(xs) - 1.5, max(xs) + 1.5)
    ax.set_ylim(min(ys) - 1.0, max(ys) + 1.0)

    ARROW = dict(arrowstyle="-|>", color="black", lw=2, mutation_scale=18,
                 shrinkA=0, shrinkB=0)

    for d in derivations:
        cx, cy = coords[d["conclusion"]]
        premises = d["premises"]
        top = (cx, cy - 0.35)  # just below the conclusion label

        if len(premises) == 1:
            px, py = coords[premises[0]]
            ax.annotate("", xy=top, xytext=(px, py + 0.35), arrowprops=ARROW)
            continue

        jx, jy = cx, cy - 1.0  # AND junction point
        for p in premises:
            px, py = coords[p]
            ax.plot([px, jx], [py + 0.35, jy], color="black", lw=2,
                    solid_capstyle="round")

        # arc across the incoming lines
        angles = []
        for p in premises:
            px, py = coords[p]
            a = math.degrees(math.atan2(py + 0.35 - jy, px - jx)) % 360
            angles.append(a)
        ax.add_patch(Arc((jx, jy), 0.9, 0.9, theta1=min(angles),
                         theta2=max(angles), color="black", lw=1.6))

        ax.plot(jx, jy, "o", color="black", ms=7, zorder=5)
        ax.annotate("", xy=top, xytext=(jx, jy), arrowprops=ARROW)

    initial_set = set(initial_facts)
    for n, (x, y) in coords.items():
        is_goal = n == goal
        bbox = dict(boxstyle="circle,pad=0.25", fc="#FFF176", ec="none") \
            if is_goal else dict(boxstyle="square,pad=0.15", fc="white", ec="none")
        ax.text(x, y, format_atom(n), ha="center", va="center",
                fontsize=17, style="italic", family="serif", bbox=bbox, zorder=6)
        if n not in initial_set:
            ax.text(x + 0.45, y + 0.12, "✓", color="red", fontsize=16,
                    weight="bold", zorder=7)

    ax.set_title("Forward Chaining Approach", fontsize=18, weight="bold",
                 color="#172033", pad=12)
    plt.tight_layout()
    plt.show()


# ---------- USER INPUT WIDGETS ----------

facts_box = widgets.Textarea(
    description="Facts:",
    placeholder="A\nB",
    layout=widgets.Layout(width="750px", height="150px"),
)

rules_box = widgets.Textarea(
    description="Rules:",
    placeholder="A & B -> L\nB & L -> M\nL & M -> P\nP -> Q",
    layout=widgets.Layout(width="750px", height="200px"),
)

goal_box = widgets.Text(
    description="Goal:",
    placeholder="Q",
    layout=widgets.Layout(width="450px"),
)

button = widgets.Button(
    description="Run Forward Chaining",
    button_style="success",
    icon="play",
)

output = widgets.Output()


def run_forward(b):
    with output:
        output.clear_output()
        try:
            facts = [parse_atom(l.strip()) for l in facts_box.value.splitlines() if l.strip()]
            rules = [parse_rule(l.strip()) for l in rules_box.value.splitlines() if l.strip()]
            goal = goal_box.value.strip()

            if not facts:
                print("Please enter at least one fact.")
                return
            if not rules:
                print("Please enter at least one rule.")
                return
            if not goal:
                print("Please enter a goal.")
                return

            forward_chaining(facts, rules, goal)
        except Exception as e:
            print("Input Error:", e)


button.on_click(run_forward)
display(facts_box, rules_box, goal_box, button, output)

Textarea(value='', description='Facts:', layout=Layout(height='150px', width='750px'), placeholder='A\nB')

Textarea(value='', description='Rules:', layout=Layout(height='200px', width='750px'), placeholder='A & B -> L…

Text(value='', description='Goal:', layout=Layout(width='450px'), placeholder='Q')

Button(button_style='success', description='Run Forward Chaining', icon='play', style=ButtonStyle())

Output()